# Signal Hunters R2 - run 20 (wave 2, teammate A): CNN v4 LEVEL + real-PKS gain jitter (train + honest validation) + 2x canvas + PL4 + DOUBLE ETI injection - EfficientNet-B0 x2 seeds (~3.3 h)
GPU T4 x2, input: signal-hunters-r2, Internet ON. RUN WITH: Save Version -> Save & Run All (Commit).

In [ ]:
%%writefile /kaggle/working/cnn_r2.py
"""Signal Hunters R2 - CNN on the 'global-time canvas'.
Each cadence -> one 128x128 image: the 6 panels placed on ONE time axis including the slew gaps
(panel p row r at t = p*(16+gap)+r), so a true ETI drift is a single straight line through the 3 ON panels.
Input channels: z (bandpass-flattened, noise-normalised), zr (z minus per-row median: kills broadband rows),
on/off indicator (+1 ON rows, -1 OFF rows, 0 gaps), valid mask.
Domain shift GBT->PKS is attacked with augmentation: frequency zoom (PKS drifts look ~0.64x as steep),
SNR loss, dropped rows, broadband glitch rows, per-panel gain, frequency shift/flip.
Usage: python cnn_r2.py DATA_DIR EPOCHS SEEDS [arch=res|timm_resnet18|timm_effb0] [PL]
Writes oof_<tag>.npy (18000x9) and test_<tag>.npy (10000x9) to ./out/"""
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as Fn, sys, os, time, math
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
LABELS = ['eti_candidate', 'frb', 'pulsar', 'scintillating_star', 'satellite_pass', 'radar_chirp',
          'rfi_intermittent', 'rfi_persistent', 'noise']
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
UP = int(os.environ.get('UP', 1)); LVL = int(os.environ.get('LVL', 1)); P_ETI = float(os.environ.get('P_ETI', 0)); P_TRAP = float(os.environ.get('P_TRAP', 0))

def normalise(X):
    x = X.astype(np.float32); m = (X > 0); x[~m] = np.nan
    x = x - np.nanmedian(x, axis=2, keepdims=True)
    s = 1.4826 * np.nanmedian(np.abs(x - np.nanmedian(x, axis=(2, 3), keepdims=True)), axis=(2, 3), keepdims=True) + 1e-3
    z = np.nan_to_num(np.clip(x / s, -10, 60))
    return z.astype(np.float16), m

def load(D, split):
    X = np.load(f'{D}/{split}_cadences.npy'); meta = pd.read_csv(f'{D}/{split}_meta.csv')
    return X, (X > 0), meta.slew_gap_rows.values.astype(np.int64), meta

def norm_t(x, m):
    """FIXED normalisation (v2). Old version subtracted each channel's median over time, which ERASED every
    non-drifting line (persistent RFI, non-drifting traps): a 10-sigma persistent line became 0.4 sigma.
    Now: bandpass = per-panel time-median spectrum smoothed ACROSS FREQUENCY (median, 9 channels), so narrow
    lines survive. Channel 64 (GBT-only centre spike, absent in PKS) is blanked in every cadence."""
    x = torch.where(m > 0, x, torch.full_like(x, float('nan')))
    x[..., 64] = float('nan')
    spec = torch.nanmedian(x, dim=2).values                                   # (B,6,128)
    spec = torch.where(torch.isnan(spec), torch.nanmedian(spec, dim=-1, keepdim=True).values, spec)
    pad = Fn.pad(spec, (4, 4), mode='replicate')
    band = pad.unfold(-1, 9, 1).median(-1).values                            # smooth bandpass
    r = x - band[:, :, None, :]
    f = r.flatten(2); med = torch.nanmedian(f, dim=2, keepdim=True).values
    s = 1.4826 * torch.nanmedian((f - med).abs(), dim=2, keepdim=True).values + 1e-3
    z = ((r - med[..., None]) / s[..., None]).clamp(-10, 60)
    # v3: LEVEL channel = what the per-panel bandpass removed, relative to the OFF-panel bandpass.
    # A scintillating star brightens the whole ON panel; the per-panel band absorbs it, so z alone cannot show it.
    bo = band[:, [1, 3, 5]].median(1, keepdim=True).values                    # (B,1,128) OFF bandpass
    so = s[:, [1, 3, 5]].median(1, keepdim=True).values                       # (B,1,1) OFF noise scale
    E = ((band - bo) / so).clamp(-10, 30)                                     # (B,6,128)
    return torch.nan_to_num(z, nan=0.0), torch.nan_to_num(E, nan=0.0)

def byte_aug(x, m, k_lo=5., k_hi=30., p=0.8):
    """PKS-like receiver noise in the POWER domain: power *= Gamma(k)/k  -> skewed log-noise like the test
    telescope (measured: noise MAD 15.6 vs 11.9 bytes, more negative skew). Also extra dropped/glitch rows."""
    B = x.shape[0]
    k = torch.empty(B, 1, 1, 1, device=x.device).uniform_(k_lo, k_hi)
    g = torch.distributions.Gamma(k.expand_as(x), k.expand_as(x)).sample().clamp_min(1e-4)
    sel = (torch.rand(B, 1, 1, 1, device=x.device) < p).float()
    x = x + sel * 24 * torch.log2(g)
    drop = (torch.rand(B, 6, 16, 1, device=x.device) < torch.empty(B, 1, 1, 1, device=x.device).uniform_(0, 0.03)).float()
    gl = (torch.rand(B, 6, 16, 1, device=x.device) < 0.01).float() * torch.empty(B, 6, 16, 1, device=x.device).uniform_(10, 50)
    x = (x + gl).clamp(1, 255)
    m = m * (1 - drop)
    return x, m

# ---------------- GPU-side augmentation + canvas -----------------
def augment(z, m, train, E):
    """z,m: (B,6,16,128) float on device"""
    B = z.shape[0]
    if not train: return z, m, E
    # SNR loss + extra noise (PKS noisier): z' = a*z + sqrt(1-a^2)*N
    a = torch.empty(B, 1, 1, 1, device=z.device).uniform_(0.75, 1.0)
    z = a * z + torch.sqrt(1 - a ** 2) * torch.randn_like(z)
    # per-panel gain on the signal part
    z = z * torch.empty(B, 6, 1, 1, device=z.device).uniform_(0.8, 1.2)
    # frequency zoom (compress/expand channel axis), p=0.7, s in [0.55,1.1]
    sel = torch.rand(B, device=z.device) < (0.15 if not LVL else 0.0)
    if sel.any():
        s = torch.ones(B, device=z.device); s[sel] = torch.empty(int(sel.sum()), device=z.device).uniform_(0.85, 1.15)
        xs = torch.linspace(-1, 1, 128, device=z.device)
        gx = xs[None, :] / s[:, None]                                   # sample positions in input
        grid = torch.stack([gx[:, None, :].expand(B, 96, 128), torch.linspace(-1, 1, 96, device=z.device)[None, :, None].expand(B, 96, 128)], -1)
        zz = Fn.grid_sample(z.reshape(B, 1, 96, 128), grid, mode='bilinear', padding_mode='zeros', align_corners=True).reshape(B, 6, 16, 128)
        mm = Fn.grid_sample(m.reshape(B, 1, 96, 128).float(), grid, mode='nearest', padding_mode='zeros', align_corners=True).reshape(B, 6, 16, 128)
        outside = (gx.abs() > 1)[:, None, None, :].expand(B, 6, 16, 128)
        # bilinear interpolation shrinks noise std -> renormalise per panel, fill outside with fresh noise
        zz = torch.where(outside, torch.randn_like(zz), zz)
        mm = torch.where(outside, torch.ones_like(mm), mm)
        sd = zz.flatten(2).std(2).clamp_min(0.3)[..., None, None]
        zz = zz / torch.where(sd < 1, sd, torch.ones_like(sd))
        z = torch.where(sel[:, None, None, None], zz, z); m = torch.where(sel[:, None, None, None], mm, m.float())
    m = m.float()
    # frequency shift (roll) and flip
    sh = int(np.random.randint(-30, 31)); z = torch.roll(z, sh, -1); m = torch.roll(m, sh, -1); E = torch.roll(E, sh, -1)
    fl = torch.rand(B, device=z.device) < 0.5
    z = torch.where(fl[:, None, None, None], z.flip(-1), z); m = torch.where(fl[:, None, None, None], m.flip(-1), m)
    E = torch.where(fl[:, None, None], E.flip(-1), E)
    E = E + 0.15 * torch.randn(B, 6, 1, device=z.device)                      # per-panel gain jitter on the level
    return z, m, E

def up(C):
    return Fn.interpolate(C, scale_factor=UP, mode='nearest') if UP > 1 else C

def canvas(z, m, gap, E=None):
    """(B,6,16,128) -> (B,4+LVL,128,128) global-time canvas"""
    B = z.shape[0]; H = 128
    zr = z - z.median(-1, keepdim=True).values
    C = torch.zeros(B, 4 + LVL, H, 128, device=z.device)
    p = torch.arange(6, device=z.device); r = torch.arange(16, device=z.device)
    t = p[None, :, None] * (16 + gap[:, None, None]) + r[None, None, :]          # (B,6,16)
    t = t + ((H - (6 * 16 + 5 * gap)) // 2)[:, None, None]                       # centre vertically
    bi = torch.arange(B, device=z.device)[:, None, None].expand(B, 6, 16)
    sign = torch.tensor([1., -1, 1, -1, 1, -1], device=z.device)[None, :, None].expand(B, 6, 16)
    C[bi, 0, t] = z * m; C[bi, 1, t] = zr * m
    C[bi, 2, t] = sign[..., None].expand(B, 6, 16, 128); C[bi, 3, t] = m
    if LVL: C[bi, 4, t] = 3 * E[:, :, None, :].expand(B, 6, 16, 128)        # x3: same scale as z
    return C


def inject(z, m, gap, yb, p_eti, p_trap):
    """Physics-based synthetic signals (labels follow the priority order):
    - ETI: a narrow drifting line on ONE straight track in global time, ON panels only -> label eti_candidate
    - trap: ON-only lines that do NOT connect (random offsets) or do not drift -> rfi_intermittent
      (only injected into noise / rfi_persistent cadences, whose priority is lower)"""
    B = z.shape[0]; d_ = z.device
    t = (torch.arange(6, device=d_)[None, :, None] * (16 + gap[:, None, None]) + torch.arange(16, device=d_)[None, None, :]).float()
    tc = t[:, [0, 2, 4]].mean((1, 2))[:, None, None]
    ch = torch.arange(128, device=d_).float()[None, None, None, :]
    on = torch.tensor([1., 0, 1, 0, 1, 0], device=d_)[None, :, None, None]
    r = torch.rand(B, device=d_)
    e = (r < p_eti) & (yb != 0)
    tr_ = (r >= p_eti) & (r < p_eti + p_trap) & ((yb == 7) | (yb == 8))
    if e.any() or tr_.any():
        drift = torch.empty(B, device=d_).uniform_(0.15, 1.3) * (torch.randint(0, 2, (B,), device=d_) * 2 - 1)
        c0 = torch.empty(B, device=d_).uniform_(20, 108)
        amp = torch.empty(B, device=d_).uniform_(0.6, 3.0); wid = torch.empty(B, device=d_).uniform_(0.5, 1.2)
        centre = c0[:, None, None] + drift[:, None, None] * (t - tc)                      # (B,6,16) connected track
        kind = torch.randint(0, 2, (B,), device=d_)                                       # trap type
        off = torch.empty(B, 6, 1, device=d_).uniform_(-40, 40)                           # disconnect offsets
        tr_centre = torch.where(kind[:, None, None] == 0, c0[:, None, None] + off + drift[:, None, None] * (t - t[:, :, :1]),
                                c0[:, None, None] + 0 * t)                                # non-connecting / non-drifting
        cen = torch.where(e[:, None, None], centre, tr_centre)
        line = amp[:, None, None, None] * torch.exp(-0.5 * ((ch - cen[..., None]) / wid[:, None, None, None]) ** 2) * on
        sel = (e | tr_)[:, None, None, None].float()
        z = z + sel * line * m
        yb = torch.where(e, torch.zeros_like(yb), torch.where(tr_, torch.full_like(yb, 6), yb))
    return z, yb

# ---------------- model -----------------
class Block(nn.Module):
    def __init__(s, i, o, st):
        super().__init__()
        s.c1 = nn.Conv2d(i, o, 3, st, 1, bias=False); s.b1 = nn.BatchNorm2d(o)
        s.c2 = nn.Conv2d(o, o, 3, 1, 1, bias=False); s.b2 = nn.BatchNorm2d(o)
        s.sk = nn.Sequential(nn.Conv2d(i, o, 1, st, bias=False), nn.BatchNorm2d(o)) if (i != o or st != 1) else nn.Identity()
    def forward(s, x): return torch.relu(s.b2(s.c2(torch.relu(s.b1(s.c1(x))))) + s.sk(x))

class ResNet(nn.Module):
    def __init__(s, w=40):
        super().__init__()
        s.stem = nn.Sequential(nn.Conv2d(4 + LVL, w, (5, 3), 1, (2, 1), bias=False), nn.BatchNorm2d(w), nn.ReLU())
        s.body = nn.Sequential(Block(w, 2 * w, 2), Block(2 * w, 2 * w, 1),
                               Block(2 * w, 4 * w, 2), Block(4 * w, 4 * w, 1), Block(4 * w, 6 * w, 2), Block(6 * w, 6 * w, 1),
                               Block(6 * w, 8 * w, 2))
        s.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(16 * w, 9))
    def forward(s, x):
        h = s.body(s.stem(x)); return s.head(torch.cat([h.mean((2, 3)), h.amax((2, 3))], 1))

def make(arch):
    if arch == 'res': return ResNet()
    import timm
    name = {'timm_resnet18': 'resnet18', 'timm_effb0': 'efficientnet_b0', 'timm_effb2': 'efficientnet_b2', 'timm_convnext': 'convnext_nano', 'timm_effb3': 'efficientnet_b3'}[arch]
    try: return timm.create_model(name, pretrained=True, in_chans=4 + LVL, num_classes=9)
    except Exception as e: print('pretrained download failed, random init:', e); return timm.create_model(name, pretrained=False, in_chans=4 + LVL, num_classes=9)

def predict(net, Z, M, G, tta=True):
    EE = None
    if isinstance(Z, tuple): Z, EE = Z
    net.eval(); out = []
    with torch.no_grad(), torch.autocast(dev, enabled=dev == 'cuda'):
        for j in range(0, len(Z), 512):
            m = torch.from_numpy(M[j:j + 512]).to(dev).float()
            z = torch.from_numpy(Z[j:j + 512]).to(dev).float()
            if Z.dtype == np.uint8: z, E = norm_t(z, m)
            else: E = torch.from_numpy(EE[j:j + 512]).to(dev).float()
            g = torch.from_numpy(G[j:j + 512]).to(dev)
            p = torch.softmax(net(up(canvas(z, m, g, E))).float(), 1)
            if tta: p = 0.5 * (p + torch.softmax(net(up(canvas(z.flip(-1), m.flip(-1), g, E.flip(-1)))).float(), 1))
            out.append(p.cpu().numpy())
    return np.concatenate(out)

def pks_like(Z, M, seed=0):
    """deterministic 'PKS-like' version of held-out GBT data for shift-aware validation"""
    torch.manual_seed(seed); out_z, out_m, out_e = [], [], []
    for j in range(0, len(Z), 512):
        x = torch.from_numpy(Z[j:j + 512]).to(dev).float(); m = torch.from_numpy(M[j:j + 512]).to(dev).float(); B = len(x)
        x, m = byte_aug(x, m, 12., 12., 1.0)
        # v2 validation: per-panel gain offsets measured on the real PKS test (std 0.13 x noise scale)
        xm = torch.where(m > 0, x, torch.full_like(x, float('nan'))).flatten(1)
        sc = 1.4826 * torch.nanmedian((xm - torch.nanmedian(xm, 1, keepdim=True).values).abs(), 1).values
        x = torch.where(m > 0, (x + float(os.environ.get('GAINJ', 0.13)) * torch.randn(B, 6, 1, 1, device=dev) * sc[:, None, None, None]).clamp(1, 255), x)
        z, E = norm_t(x, m); out_e.append(E.cpu().numpy().astype(np.float16))
        xs = torch.linspace(-1, 1, 128, device=dev) / 1.0
        grid = torch.stack([xs[None, None, :].expand(B, 96, 128), torch.linspace(-1, 1, 96, device=dev)[None, :, None].expand(B, 96, 128)], -1)
        zz = Fn.grid_sample(z.reshape(B, 1, 96, 128), grid, align_corners=True).reshape(B, 6, 16, 128)
        out = (xs.abs() > 1)[None, None, None, :].expand_as(zz)
        zz = torch.where(out, torch.randn_like(zz), zz); sd = zz.flatten(2).std(2).clamp_min(0.3)[..., None, None]
        zz = zz / torch.where(sd < 1, sd, torch.ones_like(sd))
        mm = torch.where(out, torch.ones_like(m), Fn.grid_sample(m.reshape(B, 1, 96, 128), grid, mode='nearest', align_corners=True).reshape(B, 6, 16, 128))
        out_z.append(zz.cpu().numpy().astype(np.float16)); out_m.append(mm.cpu().numpy() > 0.5)
    return (np.concatenate(out_z), np.concatenate(out_e)), np.concatenate(out_m)

def fit(Z, M, G, y, w_cls, epochs, seed, arch, evals):
    torch.manual_seed(seed); np.random.seed(seed)
    net = make(arch).to(dev); opt = torch.optim.AdamW(net.parameters(), 2e-3, weight_decay=0.02)
    bs = int(os.environ.get('BS', 128)); n = len(y); steps = epochs * (n // bs)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps, pct_start=0.15)
    lossf = nn.CrossEntropyLoss(weight=torch.tensor(w_cls, device=dev, dtype=torch.float32), label_smoothing=0.05)
    scaler = torch.amp.GradScaler(enabled=dev == 'cuda'); Y = torch.from_numpy(y)
    for ep in range(epochs):
        net.train(); perm = np.random.permutation(n); t0 = time.time(); L = 0
        for j in range(0, n - bs + 1, bs):
            idx = perm[j:j + bs]
            x = torch.from_numpy(Z[idx]).to(dev).float(); m = torch.from_numpy(M[idx]).to(dev).float()
            g = torch.from_numpy(G[idx]).to(dev); yb = Y[idx].to(dev)
            x, m = byte_aug(x, m)
            # real-PKS-like per-panel gain offsets in the byte domain (std U(0,0.2) x noise scale), so the LEVEL channel learns to ignore them
            xm = torch.where(m > 0, x, torch.full_like(x, float('nan'))).flatten(1)
            sc = 1.4826 * torch.nanmedian((xm - torch.nanmedian(xm, 1, keepdim=True).values).abs(), 1).values
            gj = torch.empty(len(x), 1, 1, 1, device=dev).uniform_(0, 0.2) * torch.randn(len(x), 6, 1, 1, device=dev)
            x = torch.where(m > 0, (x + gj * sc[:, None, None, None]).clamp(1, 255), x)
            z, E = norm_t(x, m)
            if P_ETI + P_TRAP > 0: z, yb = inject(z, m, g, yb, P_ETI, P_TRAP)
            z, m, E = augment(z, m, True, E)
            with torch.autocast(dev, enabled=dev == 'cuda'):
                loss = lossf(net(up(canvas(z, m, g, E))), yb)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step(); L += loss.item() * bs
        if ep % 5 == 4 or ep == epochs - 1: print(f'   ep{ep} loss {L / n:.3f} {time.time() - t0:.1f}s', flush=True)
    return [predict(net, *e) for e in evals]

if __name__ == '__main__':
    D, EP, SEEDS = sys.argv[1], int(sys.argv[2]), int(sys.argv[3])
    arch = sys.argv[4] if len(sys.argv) > 4 else 'res'; PL = 'PL' in sys.argv
    tag = os.environ.get('TAG', arch); SMOKE = int(os.environ.get('SMOKE', 0)); os.makedirs('out', exist_ok=True)
    Z, M, G, meta = load(D, 'train'); Zt, Mt, Gt, tmeta = load(D, 'test')
    y = meta.label.map({l: i for i, l in enumerate(LABELS)}).values
    if SMOKE: Z, M, G, y = Z[:SMOKE], M[:SMOKE], G[:SMOKE], y[:SMOKE]; Zt, Mt, Gt = Zt[:SMOKE], Mt[:SMOKE], Gt[:SMOKE]
    cnt = np.bincount(y, minlength=9); w_cls = (cnt.sum() / (9 * np.maximum(cnt, 1))) ** 0.5; w_cls = w_cls / w_cls.mean()
    print('device', dev, 'arch', arch, 'train', Z.shape, 'class weights', w_cls.round(2), flush=True)
    if PL:
        pl = np.load('out/pl.npz'); sel, ypl = pl['sel'], pl['y']
        print('pseudo-labelled test cadences', sel.sum(), flush=True)
    oof = np.zeros((len(y), 8 + 1)); oof_pks = np.zeros_like(oof); pt = np.zeros((len(Zt), 9))
    for s in range(SEEDS):
        for k, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=0).split(Z, y)):
            if os.environ.get('FOLDS') and str(k) not in os.environ['FOLDS'].split(','): continue
            Za, Ma, Ga, ya = Z[a], M[a], G[a], y[a]
            if PL: Za, Ma, Ga, ya = np.concatenate([Za, Zt[sel]]), np.concatenate([Ma, Mt[sel]]), np.r_[Ga, Gt[sel]], np.r_[ya, ypl]
            Zp, Mp = pks_like(Z[b], M[b], seed=k)
            pb, pp, pte = fit(Za, Ma, Ga, ya, w_cls, EP, 100 * s + k + int(os.environ.get('SEED_OFF', 0)), arch, [(Z[b], M[b], G[b]), (Zp, Mp, G[b]), (Zt, Mt, Gt)])
            oof[b] += pb / SEEDS; oof_pks[b] += pp / SEEDS; pt += pte / (5 * SEEDS)
            print(f'seed {s} fold {k}: F1 {f1_score(y[b], pb.argmax(1), average="macro"):.4f} | PKS-like F1 {f1_score(y[b], pp.argmax(1), average="macro"):.4f}', flush=True)
            np.save(f'out/oof_{tag}_partial.npy', oof); np.save(f'out/test_{tag}_partial.npy', pt); np.save(f'out/folds_done_{tag}.npy', np.array([s, k]))
        f = SEEDS / (s + 1)
        print(f'== after seed {s}: OOF F1 {f1_score(y, (oof * f).argmax(1), average="macro"):.4f} | PKS-like {f1_score(y, (oof_pks * f).argmax(1), average="macro"):.4f}', flush=True)
        print('per class (PKS-like):', dict(zip(LABELS, f1_score(y, (oof_pks * f).argmax(1), average=None).round(3))), flush=True)
        np.save(f'out/oof_{tag}.npy', oof * f); np.save(f'out/oofpks_{tag}.npy', oof_pks * f); np.save(f'out/test_{tag}.npy', pt * f)
    print('DONE', flush=True)


In [ ]:
import base64, os
os.makedirs('/kaggle/working/out', exist_ok=True)
open('/kaggle/working/out/pl.npz','wb').write(base64.b64decode('UEsDBC0AAAAIAAAAIQBbkSyh//////////8HABQAc2VsLm5weQEAEACQJwAAAAAAAHMEAAAAAAAAnVm7ktNAEBylfIUzQ5UDLuUDyKBICIgow5kioDjKpkiAr+CHuYdX8+qeHXvxIWt3dt7dsqR/b9+/efdhkV/ye3t7OH0+bl9ttn8+3Wx3m+2Xu+PP4/77x7vj7eFh/vX+2+lwP3/6uv9xuD9/fvPyfuxe7DZ/N1eOZwsZwhaelh7/5OHT23MWWLci4cmMuKOYE1HlM4069SQeNgkRprNy/ueXxDu5+FmvaUTgjiRa/S4L8+7aYdNhy9O0YgrRk83SdctJSxBnRjSPmmkVDFvk/AHNkHqMQID1KHOTwAgUX8h32Os0Z+qZyuQsNMZISoU0194YpyUKuWWyA1igGjqc5YJUYAbEqMakE/aSkSaAqD2np8guE7GeDS8K1s12xQDDJ9T203psFLqiBUE9/WR/mINIWM2vzFyRjZC0hGggKaMZR6q24EkeHNGZThYoMJUQ0S9FzkVc1JJyyZiIqkQ86jTBguazGdOlvchH05A1gEwLD/8AEz+ky8EeaGgDOygCGJBoRrJPgc2tI1LkMzWyZWpaZ2B+eIpFz46gAsZCoUtKtKdQLtBkTw07kBwrpVEWsUXIP/zQGeJBHwT0JuACBMTCZjSE3POBTuDvHMYD5h3xJa4uVLLATPvAnOczv3IP6eQ8NCpBmuFxhmjHOb7EKCK5jooZc3ntyO8oXwGGDUEHbzqKS88iJruh2FCB0z1xtFCK8gHViVuUsG/4Ltlk7WDpN3a5tZszcVTu2t/+Pc7ZO691CruKybdgWDJTz5eNNEsV4UARW8Cm0mJHyDRpP7tX4goQw1d860pnuG2S4WRvsLMzcWoSmqxmLnEOt0z6IdXSNYKkv4dSLOYQ6sAav6WYwMejS3/wU/2Bf64ZsItoma+yVIHiQjWItQqbBSeM1GkK05XEnvp1f2sJsxWuBaWb9eixUCTrsGoAACVSd1vJdCdd8bhaCvPxSlPSv1TRpDVIN6sMEMy1JuStKu3PP/9sGDwQQfonFtB5vq8uklLBrAJ0Kgzc4rwX/6k8KEfMfA80hT62s9VHoUApEVZ5oEpkckpBS7Q3LreT0CFlL6NHNSQBudVeTO5OWQripMnIPCixPZicVSiXyl3pC3KodcxmagmaWi9DGkJyqLS1xromraqexy2arnZiKUkCUxJF4VTgcOJVOQl3WgeBWkqiwU1Oh3E+u0o6JOwmr16A7220QUV6llzgO6GY2R+wNvQTiorSfmk8VvQXJI9rsFmATrLmtKArcu6zVSZFWvSdgEaKI7dGXVQfY/RbwHdEhlZX+ZZl/G9EY2Zx8mLeygFE5gkjfJZnPMz8tMQzTBPIgOlRU+jSE7I8r7oK9yC/StpwCeTYjJh0TArk0yQgs0sNQuZUXLQm4MvZCdYTgUD4Zfg79GOp3HJZt/rAgeOk0kkCH4kCjynO1CucQzGfjvYskPeuTIGz2DFBmK10xPtEN5mEV69Q2Rwqtn+AGiJOyWnGhlEKbzdDuO5x7wShwKbpB/CqnekAPYBfwWX2SI8UnA7z9T9QSwMELQAAAAgAAAAhAOfOWef//////////wUAFAB5Lm5weQEAEAAoFgEAAAAAAAcYAAAAAAAArdzPCiRZEcXhnGlnHHyK3rXCbIVGXLtT3LhwJYPToiCO9Igb8Sl8YUecuznw8bsl5qaorKzM+yfixIkTUfWvX/3ml7/+7SfP359/vPv6w7e///juZ2/f/fxP7999+fbdH775+LePX/3ld998/PrDf87/4qs/f/vhu/Pf/vGrv3747v2P379//9Mvf/Ll23++/V+PH/3g+e/x+fevn37/+ub71x/O6xdz/efz+Wdz/lz/aZz/Yl73vju+c91n87rjPd878/x03u91X8znn8zrzkP33/nsOu99zzh23Xc9zvWfzft97s5vP9f1b+bz3Yed145P99O+nvPnOPPa9d/13vvsPp7v7Ti1f7sP9b0d1z5v113jP98rP9S8dx93/Htf7dvue+2v/Gvf7/zO/pa/r5+uXex6LO7s+fUX7cfu9+7Dzl/z23WUfxSe7ngX5/S887p4sfalfT3zXXwRXmo/Ne7d38W9fb/3e3B+/XjHWTgiHF+73fc7Hu3/2ovwcvdd/qzXtfu147XX3Z/Fs53v+kfxA8Xzna/8Q/G++ILwZuex673+VXYv/Nh1Xn8snnIO4fP6k+Lgjk/rsvu6+6Lzdb+1j/U32ff5vuxZPGfXUzz2vG6cXfwUD9jxa13Wr3ad6/57fuPhjkPxbuPy4tfGubVvxY3FtV3n4jF73a6X+NziZvFV8YWdh3hB8frz/pnr9nm3fHVxSLxT8Wr9pPBA8UDrp3i1r8KB4lnib8pzdh81fu23+Mh+b8epPKfeLw4qHorXVVwS7ipOi5fucxUHCvdlH8+8F09cHF/8kx2Kv+y6i6dWnNa4134Vxys/0PwXH8qPtR/Sb/R+7X3X4ZnzyhsV58X/zvf3/vv88l+t36vxYtej+O767T5XPGHtXXlG+dl+r+JC2fOtXlD7v36141L+r/195n3lpXvdOYTHxWPWTjevFQ9Wfii/Fy6XvrD7Kb6qOFl8Xesruyi/kL3e6sqbT4lfiH+Lh1beWHlTrau+fw7tq+yzeLH8c+2ynqM4vPmw+F7lRWun0jWUZ22c2rhc9YzFpf2e4tK+Ko5KX5HeWHi4+yY+rVfdX/ctfFW+Ln8oPWntWnqwdM36ntZTuC9+v/aqeLT2dY61i13fve/iyBtct+u+67J2Uut4y392PRdndf+KJ+fYuFG4tveRf4m3CQd1f/mf7E56lHC77FFxo/IT+bF4S8V98YDKg0oXEJ8s/lFxQHlG1eNKFxB/qfz3VufQOt3ajfB846/ydvFe6Y6KXxv/bscrPJEupTq78gLtm/SBHdfaq/iHeJHwS/XptRfxroqv4i2qVwrvFdfPof1c/ibdXXYuO1Xc0TqWziJeqPym9BjxLvn7fu+Z95qfeJ7yJuFI6QZr/9JpnjkvO911UJ6966/8a8epfh3ZqfxMeZnmd8svZJ/Cjar/7H5Kz1BcqPijOsCuq/pJtP+qj1X9V/fbdRGOic/VuKpvqnhh1SWrXlNxqHSgHbd079KHlJ8o3pX9i5/Kv6vuWHgvP1LdsPqcqm+n+I3yzY0H55Adal+Lh1RcLV1PvLNwYHFe+azwQP5WesjyxN0n4VrpT3v/4huVL609VN50Du1j8SbNu3Cq8tLS8XZe4kF73/2+6k3LRxXHSt9cu33mvXBT+CJ+KrupOsU+57m8bu1b49P1uk64IT6hfdn7bB6hcek5qjuoXnObf0gfLry41Um3zlA4UzrmOV7lTeKrOi/9TbxA9qp11HPl/8KripPKE3ccwjXtk+KT7Ft2JL+5rd9oP0oH0ufiT1UHKj3mwXnFn+K36g/b8ZxDcVw8Z8fz4LqqJz3z+c7rmfNrX2sP4mfyX9WvZffCobI31WVqnYvfSI8Rn9fz9/s7jmc+/3zOl96lunvp9s9cJ11AOCt+sPgr/9/7VP+WeM3aqXShqj+JJwgXlYdKl1Cc0nnlK+uPpRdKl1Iess8pvUF2rXivcYjviS9IR5X/rZ1pnRWnz6F6ney9+rPUd1A8vupIlSdo37QvpQeLT9zGu+Kn0o2lm1aeVvh2jsp3FQf2c+H04pZ0nurb2fV55nPpzxVX1d9RedKtXVd9R7/rEA+8rd9U/i17Eq6oj1R8oni9+Jb4mOxL9616h/Rn5Qelcxbe3vK92/FXXUd168W56vPRvGWfu0/PXKe6Z+ktOx7p/+JBpeMvr9J4VF9UfKg87FbnUr4t3qR8s+qIz7yv/gDZy+KF+Hzpp9WnWvUT+Z/yF+2P9ld6rfRq6Vel360/rb3Xe+GOeI74g9ZNvFrrqfhyDumG4slrLzsexa3KC4Wziruq3yjPuO3/2nUUH1s73OfuulWdVX53m/8oLhdOL04pj1UcEb9QX4/WT3FGvEx2Jp6kOkzpGrsPwk3FFeG4cFrzkJ8Jf+T/z5y/rR8p/1I+qHgofePWf7UP4ou3epZ0H+GS4pDsWnrTjkd5m3iCnl95svhs9QOJp0o3Ut2rcFD5uPhLvSquLm5WvVR5o/Qa4Yr41q6z9DnZ3219ZfdV9VvNS+OSrlg65K3+Jh3kHKWfaV8Up4UX0uG0PsW3pd+UH5dd7edrX7KX4h+Vf8tPbnWYXRfx5o2jxSN2HsVDqs/vwXXiIeLx+7p8RXUF+bn2WfmW9kP6w/IC+ZXOa7+feb/f333addHv/7QO2hftp8YtPWPtUnhZuoV0o9I9Fa+lvysOqV4n+656Wek+i4PKR8Vz5R+yC82j8oXbvnqtv/p/1U+pPifxEO3/4o34rXDi1TxW/PK8PvO+eOjuz95f9qX1Uzy41XFKv1x8Kv9XXlDj33xXeY3qAOWvioOv9jkoT3hVn1y8Ur61caPqVuqjeNUvNp5U3BWvLD/d7y8/UT6u793WJcrPFLfWrmt9Sv/XOigPLf1W+s3isuJh5eEap/S24h+Vr8lvVK+TvWm/Zae38V3xSXWM0sE1/nNo/84hO611lc4gPW3PL25U3iS8VT1PfQyKW8Jf9eeLD2kd63PFR43rFk/lN8ID2fWr9q58QHYjnUfrI91c/rLPUV69ean4jHhoxR/Zc/UH7jiUp0o3qvqM8g/VExSP9vOqZ1S8K38T/hQOLd5Iz7vV4/f7O26NV/XQ9c/yi53POXYe8n/F5Y0DlRfvOCuPkd2J9+84nzm/4zyH+Jn4i/Jb1XcqLosnineeY8d5Wzcu/rjXq++l9BjNW/Fb+mONW/UV4Y/ym6rjFS9Xf1iti+ZTcV/9DNUPJDvUflf+qftKx634Lz9aO6/6p/oqKi4If3d+hUPSH6Q73OpLt3xO9qM4L9yvuKg+h4rH0iWkAwrfFX+qjlm6mnBO6ykdR/sk/r37IV3gdp3Pofxl11+4t/ORLqzfeeu5slfpcqq/KK7e9q+UXr5xZudfurt0IeHq3m9x+Tbuyz6r7qJ9rrxV+qZ0gl2Hc7z6+xLhv8YtvNn8Rvlk9RHUuHd9C49LH6r8Tnxe+CZ837iuvEH9S8WzlI9oXspjnnlfOC4+IjvefEp+Jd1B+1c4sXYjHfv295u7v8X/d/5lN5V3yx6Kr6leJVypOFz2ITvbdRNu1PrWPKWPlD59m5cqHm08v81TK29T3Kk4IT1BeZz8qeJI9UHsOPc67b90BfES1b9v7Vo8b/ep6smVZ4jHLo5qX6vPSOteuCgcFi7JzlSvEX6s/0jfqzz2licKf8VLd957v1tdW30Ysq+q/6rvTPFG+CC/l/6w+1R8Rv1B2ued7zPva52F67f6rfBPdrRxZ3mx7GDtofxmP9c41c+n+Lw6QfEV2aPip+rnu1/aV81POovya/mV1ku6qHQS8UzhfOVJOq9+qcrPi1e8Wm/bPHRxeMdfdeuNj3tf8ZV6jvy95rv32/tofRUPqn6uuKR6TtlV6caav+pRr+bXpX+o/iRcEB+RfcvPS48XzsvvlY/r/hvXdv2kz67diH8JpxQfhauVV2t/1fe1+1N8ZfmN6qDSc6o/rvjRbT6j9VL+qfy2eJDypZqX/Hw/V/1A/WyVl8nfpUfd8nzhs/xTz5VOXn5TdisdVLqWnl/9PMtzlJ8ozlc/0jOf136Ih5Sfls6u/dh11/831XXKH8uP1y9kf8s/l6+eQ364OHSO6qMpO5XuUL+Xlv1oHcVjtU/KJ5QnSV8X/ui11qd4uXBV/Q67Thq3/K50NPnRM+/FR9a+xT9l77VOFefUB1nzq7hZ8Vi4VPmc8gXx01s+W3a59qbxPvNefqP71jouvt/WF2/5mPJ8+YfqHaq3VZ9mxccdt+Kv6gPVH6F6YvVjFP5WfiO+pPUX7suO9F78W3Ul8WrZp/oiSi8Unq5dqF679rb9EqXTVj5ScUv1EvmF7OV2HxXH1V8q/LrVtYTbstfbOK55S59T3l7xTvm94lblc5UX3+oaiu/rT9UnsnhefWC3+V/lC6rfl31LF1l/Vz+D7KXGIRyuOLJ60o5b+yk80e8ja/92fOfY+yj+K7+s72v+wgvpA3u/xbHq91JcLFySXiA817yEc4sPirs773NoXcT/Kt9XPK88rtZ965SrS+74pVve6jDKX9dfFIeLV5Rurefvc0tnOIfwvf5v9lbPkf5QfHVxXDx196v6JGVX8hflA5WnV/5cdnZbv9j7veo/hfPSa4U74lvSCxXXxQ9lJxUPFF/V96Dvi4frtXhV5cNa9+Ip1S9T81Cd49X1rTqb6pm6n/y1/EV84pn32rfSvateXPXDyttv45JwWXxD+LXjUh+t9G/1W0mnUR/gq33IVVet/5WV3VXc0GvpK8J9/U5MOrLsoeoZt/0kqouJv0p/UVx+5v1tXBQvrf6oqusITwqfhKNrh6/qhuIb2ufii9V/on3Wq37fJ14v/UR+d1tP3/tqv2o/dl3387WP+v2B4rN4k+yq9OPiH6VfVN208v+d523dXfnmfl46jeoSqjMovlT+u/YpnC/eI/+v+Ka+R/XDa58qPkk/Ufy6/f+14rfiGaUn3PKEzb+FJ4qf2teKB/u8sqO1Z+G/4pv2WX3wsp/yW+Wd4s3Kp8Qn5DdaJ9lZ4ZHi0a7LM+fPsfMpu618QfakfE489Rzyf/0fgda/6i+Ko+K/mo9wuup2pe/t96rvVn76zHXio8qjKh4I38r/Fo/1uzWt1677OZS3Cxc0jtv9KD5ev6NS3K28UvtRfEB1zx2v8rqqU4i/iMcVn6l90+8qdv8Ub5TPC7eeuW7xSP4pvFm8kF3V+G51OPHgsj/FC42j/LjiSfUn7TiLRwnXq+4rXiC+InwTXkt/Up1D+abin3hm9aX/v/qmFi8rHhfvuM3Pdv1L/73VMcRnZR/iU+Lp0j32e6ov7X3Uhyu+Ij1Idl/4rPqN1uMcG4d2vYTf4geKX+JV0qcV33Z8wvfKK8Wbqt6k+rZwVuuofa38UjhX9fNz7PjPUeuw8b/2rXTJ2u9nPleeUvFcdqBxaNz7nPr91K6b7El2dI7b/teqOxV+Kq7selafjvL23bfSd555v/ry4lj9LrjsUP5dPFH8X/FI4xfPPod0ndJJNL/9vvi99nd1LO2/6t7nUFxSXan6qbSP4mOlmyluK3+T7r/z0e8bi6+dQ3FOuFX8U7qn6jWVH2v9isfKr4p3aR473tKtS19bPxb/037t/ITfu3/Ke9fPVDeo79/WCQp/tI5ar+WtWj/pQpVXij+p7rjrJl4jPb7sqvSywk+N5xzSWcQzbusJi1uqi5ReVX1Gis+Vt66uqHyo/PxVPlg6T+Ufu++7vvV7R623/EN1Q+mgpX8urpyjdMLKA9YOKo6vv5Qd6vkPzut69ddWvVf5efX13Oo68nvhU+Wzaz/Kx0qfFx5sPe32d8xan2fO73qcQ/Z4m8fr+3u99OSaZ8UN6Y3qZ9I+im+fQ/XBV/MG5b8bn2W38r+yL+Vj4pvqY1IfgJ5TdUDlyaX3Kf+6nYf+/0N+VnaqeK58WftQ8y/c23mcY3GmeNbG45qv1l2fF7+u+qR4/caNV/uV1Be7fF24KL2h8HDPbxyXPYiflT4gHa1+7yL/kJ5f9rr30Xyr7qZ8ULpi8S3p7WVnwqHy38rbCyfOUTqG/EjjFG4o7squ1Gcqnb9wcu1fvFG6bfFT5ftaN+lj4nnSJyrerb1K3608oXBcfqb/+yu81H6IR52j9CXxt9v93HUqnlnxS3nL8m49t+rE8k/pXuK7ex/FveUR+n30q/mZ6gqqU6/fKG4oP6m6S+FNxWHplfIP3f+Z61S3k52VniVesuuveSiuqA5T/P7VOtLi6I67dO6Ke+vfe2j9Kl9VvUD5VOl/Zb+rB64fl94h+658Vvsk/N/8Svgrv77FOfG64inKq6pv7ZbPCq8X96rvr+a9dq37lD7war4unljrU3xI+qrsSPyi5rvXVV5S/Kz6FVUvFj7f5rvij8X3nzl/m9dVXC1++qpeoPiovjz5zTkUZ8W39ns73nMUXiqeiM/se8XF9deKz+sPz5yXHalfSNfrvhV/tG6Ff/KT+t8H8YldP/W7K+5Uf5TmW7xFee0z5wtvtc6Kn5W/n6P0iH3ujnvno7qk7EB9qcIx2U/13e3zlT8X/7qtK8g/pVvu98ufyo6Fq8WHi7crXgn3VAeSDqQ+2+or1D5X/JMerPxG85EdiV/vesqvz7HzqH674m37HOUn1Y+hfOKZz7Vu0q3VV1u/U7vtP73Fffl76czbPyzcEg8o3lE4J3tXHUk8Ze1FdZRbnUb52q7HXlf+JX3pHHVf6X+3+65+LPFl2bHi7+oIpauITwuv9b3iPTtPxePq/1TfhnTnW31Cn9f/fZX/Fx8uvWvtRbpA4W3931PVv5Q/i5eoj/H2dweKO7p/9clqHbWPmp/y9X09h/xF9S/pE+KXqodrXvV76qrbigeId4svVV/mnr/tC9j4pDp43U/rpPxHeqfqmcVLVX+V3cs+dl/VN7V4r3ggey2eW7yy8teqkz1zXuMQ/pTdaj9u8+3lDWv/wvPCj+I/9XsNvZeevfZSurfijnS6iu/F77U/r/bd3eZHxQPln+LHVYc/h/qslMcrji5ua73kB7tfyv+07sX3FFdv+fA+T/xJ+yNeI75bdim7k/0pXyjdef1fOpHyUd23+qgKX4pf1roVLt/yNO2r4vXGGdXp1k5Vd16+c9tHv/6svHafcxtXSsfbcT3zXvhXdTflzbf9MaqbVF/sxhPFafmX+rfUd1P1RdnHbT17ecWuk3BP+HkO6S+1Hoozyj9k37I/va/fE8j+VKeTfez3pIvv9efY+QkXxBtv40zpCZWvqO9o7az2tfIO6TfSw3c/lPfIjsQ3hau3dfLifa/mfZqPdKGyP8Vj6eSKE7f9ILc6iuKHxnublyvvUJwsOysdofo6lI+qL0Z+XH0BhXuV/8iPi//v+3Ps/MSHK+5qPcUjbnFR8fxWr1hcqX3Q58+cl75W/F/5ytqb4knpqbf2qHE++HztRLgh/JCdaX3Eq2r/xKdu+dmr+aNwXf2JyveV7zxznfoldhwa723+p3rQrq/wvPxE46z+Q9W9xMd3XWX/4tEbbxan1z6kj0pHEN+rfK7infhC8YDi56ovVT+z+tbOUX0c0jeqj/PW/oRHqmPcrt/O//b3oOpj3fGcQ/lK5bXFa6v+Jbtdu1R+Itze6yrvKh6i/bvt41BdYK/f/RH/qHr3fk/19uqXlN1V30jFx4ovwg/llYsL4sW3eXXFJfnd9m2fo+xc45RuoOcKR/Y++lz2Vbiwz187Vt6u+rTmX7gn3Faf2cZn9WftOG/7EkuvrTxWuknpDuLFWlc9X/nGrV6l+FA8Ye2j+sD0POkpxX+LP5V+pnxf9ZrVmfc+pX9WPqn6yO0+3tb1pDeoz0D7XXztVgfYeW3+v/FNPFc6RPVzS/8WL7z1J91Pfi5c1f7WOpb+obxBPHHz23NUPWn3TbxP+d/OU/mq8EZ5v+674xTvUb4ov5PetTxMeuTa0catHX/lA+W3wvXlo6/yZOU1lVetPag+oX3S75f2vOK+9BJdV/WKvc/in+KR7Er4rvpn8cS1w9oP+Zt0QeGp1u/2ecIr5dnFV595rzqQcGzPCzcUjzUf+bP6p1VvEU7f2rXi145z7Uj7UP20GvfywcLxWx4t3UXxc/nqzvNWxxJOF95LZ648W357m88Jv6W7Ku9bvxMfLr5X/Vdrh4VfqxvsfKV7aH5aH8VLra94WeW9yrduv1d5yvrNrkPVM0pfEF/S+Fc/EL4KH8UbdP/i3eqjkE4nvxP+LE78G1BLAQItAy0AAAAIAAAAIQBbkSyhcwQAAJAnAAAHAAAAAAAAAAAAAACAAQAAAABzZWwubnB5UEsBAi0DLQAAAAgAAAAhAOfOWecHGAAAKBYBAAUAAAAAAAAAAAAAAIABrAQAAHkubnB5UEsFBgAAAAACAAIAaAAAAOocAAAAAA=='))
import numpy as np; z=np.load('/kaggle/working/out/pl.npz'); print('pseudo-labels', z['sel'].sum())

In [ ]:
import glob, os, subprocess, time
os.chdir('/kaggle/working')
D = os.path.dirname(glob.glob('/kaggle/input/**/train_cadences.npy', recursive=True)[0]); print('DATA', D)
!pip -q install timm 2>/dev/null
EPOCHS = 30
p0 = subprocess.Popen(f'CUDA_VISIBLE_DEVICES=0 UP=2 BS=64 LVL=1 SEED_OFF=27000 TAG=effb0_pl4eti1 P_ETI=0.1 P_TRAP=0.04 python -u cnn_r2.py {D} 30 1 timm_effb0 PL > out/log_effb0_pl4eti1.txt 2>&1', shell=True)
p1 = subprocess.Popen(f'CUDA_VISIBLE_DEVICES=1 UP=2 BS=64 LVL=1 SEED_OFF=27500 TAG=effb0_pl4eti2 P_ETI=0.1 P_TRAP=0.04 python -u cnn_r2.py {D} 30 1 timm_effb0 PL > out/log_effb0_pl4eti2.txt 2>&1', shell=True)
print('started')

In [ ]:
import time
while p0.poll() is None or p1.poll() is None:
    for f in ['out/log_effb0_pl4eti1.txt', 'out/log_effb0_pl4eti2.txt']:
        try:
            lines = [l for l in open(f).read().splitlines() if 'fold' in l or '==' in l or 'Error' in l or 'per class' in l]
            print(f, '|', *lines[-3:], sep='\n  ')
        except FileNotFoundError:
            print(f, 'not created yet')
    print('-' * 60, time.strftime('%H:%M:%S'), flush=True); time.sleep(300)
print('BOTH DONE')

In [ ]:
import zipfile
from IPython.display import FileLink
with zipfile.ZipFile('r2_outputs_run20.zip', 'w') as z:
    for f in glob.glob('out/*.npy') + glob.glob('out/*.txt'): z.write(f)
FileLink('r2_outputs_run20.zip')